# Reproduce the bridge ablation (full scale, real MNIST)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstiltner/collaborative-nested-learning/blob/main/notebooks/reproduce_bridge_ablation.ipynb)

Runs the real bridge-ablation training behind the **+89%** claim on
[jasonstiltner.com](https://jasonstiltner.com/projects/collaborative-nested-learning):
does adding bidirectional knowledge bridges to the CMS optimizer improve continual-learning
accuracy on Split-MNIST, and by how much, at each regularization strength?

This is full scale (real MNIST, ~12k images/task, 3 epochs) — takes about 10–12 minutes on a
Colab CPU runtime. The repo's CI runs two different things on a much faster, smaller
cadence: a fixture-based **smoke test** on every push (proves the code runs, ~2 min,
does not publish a number), and this same full-scale sweep on a **weekly** schedule
(publishes `ci_results/latest.json` and the README badge). See `CHANGELOG.md` for why
the small-fixture smoke test doesn't double as the badge source — this specific bridge
effect turned out to need close to full MNIST scale to show up at all.

In [ ]:
!git clone --depth 1 https://github.com/jstiltner/collaborative-nested-learning.git
%cd collaborative-nested-learning
!pip install -q -r requirements.txt

In [ ]:
# Full scale: real MNIST (downloaded here), reg_strengths matching the site's chart exactly.
# ~10-12 minutes on a Colab CPU runtime.
!python -m benchmarks.run_bridge_ablation

In [ ]:
import glob
import json

latest = sorted(glob.glob("experiments/results/bridge_ablation_*.json"))[-1]
with open(latest) as f:
    data = json.load(f)

print(f"{'Strength':<10} {'Without':>10} {'With':>10} {'Improvement':>12}")
for strength in data["config"]["reg_strengths"]:
    r = data["results"][str(strength)]
    wo = r["cms_only"]["average_accuracy"]
    w = r["cms_bridges"]["average_accuracy"]
    improvement = (w - wo) / wo * 100 if wo else float("nan")
    print(f"{strength:<10} {wo:>10.3f} {w:>10.3f} {improvement:>+11.1f}%")

In [ ]:
# Plot (matches jasonstiltner.com's RegularizationChart.tsx)
import matplotlib.pyplot as plt

strengths = data["config"]["reg_strengths"]
without = [data["results"][str(s)]["cms_only"]["average_accuracy"] for s in strengths]
with_bridges = [data["results"][str(s)]["cms_bridges"]["average_accuracy"] for s in strengths]

plt.plot(strengths, without, marker="o", label="CMS only")
plt.plot(strengths, with_bridges, marker="o", label="CMS + bidirectional bridges")
plt.xlabel("Regularization strength")
plt.ylabel("Average accuracy across tasks")
plt.title("Bridge ablation (real MNIST, full scale)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()